# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# My Lane as an ML Task

## Chosen Lane
Refresh / Content Opportunity Scoring

## ML Task Type
Ranking / Scoring

This project is a ranking problem rather than a traditional classification problem.

The objective is to assign every content page a refresh priority score and rank all pages from highest priority to lowest priority.

The final output is a ranked queue that helps SEO specialists decide which pages should be reviewed first.

Although a classification label such as "declining" can be used during model training, the actual business output is a ranked list of pages.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

# Target / Proxy

The ideal target would be whether a page actually loses performance in the future after a given observation window.

Since the starter dataset does not contain future outcomes, I will use a proxy target.

Proxy Target:
A page is considered declining if

trend_direction == "down"

This proxy approximates pages that may require a content refresh.

Future versions of the project could use:

Previous 90 days of features
↓

Predict decline during the next 30 days

which would produce a stronger supervised learning target.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

# Success Metric

The business goal is not to classify every page correctly.

Instead, it is to rank the most important pages at the top.

Therefore, ranking metrics are more appropriate than accuracy.

Primary Metrics

• Precision@K
• Average Precision

These metrics evaluate whether the highest-ranked pages are genuinely good candidates for review.

If reviewers only have time to inspect the top 20 or top 50 pages, Precision@K directly measures the usefulness of the ranking.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
!git clone https://github.com/dev-hashh/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 120 (delta 36), reused 84 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 1.88 MiB | 12.58 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/flyrank-ml-internship-starter


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import pandas as pd

df = pd.read_csv(
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [11]:
df[['content_id',
    'client_id',
    'impressions_90d',
    'sessions_90d',
    'trend_direction']].head(10)

,content_id,client_id,impressions_90d,sessions_90d,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,down
3,content_331d6c4de07b,client_19581e27de,11751,78,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,down
5,content_d4084a4bc775,client_f369cb89fc,3970,5,down
6,content_9a34b442b552,client_8722616204,20,1,down
7,content_a63219c6e95a,client_19581e27de,1724,28,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,68,down
9,content_c27558df2b0c,client_19581e27de,1240,3,down


# Unit of Analysis

One row represents one content page.

Each row contains observable search, engagement, and freshness signals describing that page.

The ML model predicts how urgently each page should be reviewed for a content refresh.

Therefore,

One row = One content page.

In [12]:
df["target"] = (df["trend_direction"] == "down").astype(int)

df[["trend_direction", "target"]].head(10)

,trend_direction,target
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


In [13]:
df["target"].value_counts()

,count
target,
1,16262
0,13738


Target Definition

target = 1

The page is currently declining.

target = 0

Otherwise.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

# Why ML Beats a Fixed Rule

A fixed rule might simply select pages that

• are older than 180 days
• have declining traffic
• receive many impressions

However, content quality depends on multiple interacting signals.

Examples include

• search volume
• CTR
• engagement rate
• content freshness
• average ranking position
• page age
• traffic trends

Machine learning can learn complex relationships between these signals and assign a better priority score than manually chosen thresholds.

Instead of relying on one rule, ML combines evidence from many features to rank pages more effectively.

This allows SEO teams to focus on the pages with the highest expected impact.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.